# About this notebook

In this notebook, we download the latest PDFs directly from TOME's owncloud.

In [2]:
import fitz
import requests
import io
import pickle
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
from bs4 import BeautifulSoup
import pandas as pd
import nltk
#!pip install sddk
import sddk # our package, if missing, uncomment the previous line
import requests
import getpass
from bs4 import BeautifulSoup
from requests_oauthlib import OAuth1
import zipfile
import io
import os

In [5]:
# accessing owncloud.cesnet.cz with sddk package
user = input("Insert your Username code (a long string of characters and numbers): ")
password = getpass.getpass("Insert your Password: ")
s = requests.Session() # create session
s.auth = (user, password)

In [6]:
resp = s.get("https://owncloud.cesnet.cz/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip")
resp

<Response [200]>

In [7]:
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        # List the contents of the ZIP file
        print("Files in the ZIP archive:")
        print(z.namelist())

        # Extract contents to a folder (e.g., "./extracted_files")
        extract_dir = "../data/test_pdf+xml/"
        print(f"Extracting files to: {extract_dir}")
        z.extractall(extract_dir)
        print("Extraction complete!")


Files in the ZIP archive:
['/log.txt', '1698730/Dorn1569_Artificii_chymistici_MDZ_MBS.pdf', '1698730/Dorn1569_Artificii_chymistici_MDZ_MBS/metadata.xml']
Extracting files to: ../data/test_pdf+xml/
Extraction complete!


In [8]:
base_url = "https://owncloud.cesnet.cz/"
resp = s.request("PROPFIND", base_url + "/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/")
resp

<Response [207]>

In [9]:
soup = BeautifulSoup(resp.text, "xml")
all_items = soup.find_all("d:response")

In [11]:
len(all_items)

76

In [17]:
hrefs = []
for item in all_items:
     href = item.find("d:href").text
     hrefs.append(href)
hrefs

['/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15316988.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317065.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317359.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317689.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318056.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318280.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared

In [24]:
target_dir = "/srv/data/tome/tome-corpus/emlap_raw_2025-04-08"
try:
    os.mkdir(target_dir)
except FileExistsError:
    print("Directory already exists.")

Directory already exists.


In [25]:
hrefs

['/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15316988.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317065.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317359.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317689.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318056.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318280.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared

In [19]:
resp = s.get(base_url + href)

In [26]:
base_url = "https://owncloud.cesnet.cz/"
for href in hrefs:
    if ".zip" in href:
        try:
            resp = s.get(base_url + href)
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                # Extract contents to a folder (e.g., "./extracted_files")
                extract_dir = "/srv/data/tome/tome-corpus/emlap_raw_2025-04-08/"
                z.extractall(extract_dir)
                print(href + " - Extraction complete!")
        except:
            print(href + " - Extraction failed!")

/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15316988.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317065.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317359.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317689.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318056.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318280.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3

In [28]:
len(os.listdir("/srv/data/tome/tome-corpus/emlap_raw_2025-04-08/"))

76

In [29]:
source_dir = "/srv/data/tome/tome-corpus/emlap_raw_2025-04-08/"
for dir in os.listdir(source_dir):
    try:
        print([f for f in os.listdir(source_dir + dir) if ".pdf" in f][0])
    except:
        pass

DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.pdf
Dorn1584_Commentaria_in_Archidoxorum_MDZ_MBS.pdf
Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Venice_1515_pdf.pdf
Vadis1595_Dialogus_IA_Wellcome.pdf
Greverus1599_Secretum_MDZ_MBS.pdf
Pantheus1530_Voarchadumia_ONB.pdf
Ventura1571_De_ratione_conficiendi_lapis_MBZ_MBS.pdf
Paracelsus1560_Libri_quatuor_de_vita_longa_MDZ_MBS.pdf
Mirandola1586_De_auro_libri_tres_MDZ_MBS.pdf
Auriferae_artisI1572_MBZ_Augsburg.pdf
Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.pdf
Fanianus1560_De_arte_metallicae_ONB_pdf.pdf
Severinus1571_Idea_medicinae_MDZ_MBS.pdf
Rupescissa1561_De_Consideratione_Quintae_essentie_rerum_GB.pdf
Senior1560_De_chemia_senioris_MDZ_MBS.pdf
Gessner1569_Thesaurus_Euonymi_Philiatri_Liber_Secundus_MDZ_MBS.pdf
Anon1550_De_alchemia_opuscula_MDZ_MBS.pdf
Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.pdf
Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS.pdf
Trevisanus1567_De_alchemia_MDZ_MBS.pdf
Albertus1569_De_concordantia_Hippocraticorum_et